In [38]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from collections import Counter
from wordcloud import WordCloud

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
from utils import TextPreprocessor, Word2VecVectorizer

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ruslanishakov/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
df = pd.read_csv('../data/train_v3_drcat_02.csv')

In [4]:
df = df.sample(65267).reset_index().drop('index', axis=1)

In [72]:
df.to_pickle('daigt_dataset.pkl')

In [5]:
X = df['text']
y = df['label']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Vectorization

In [7]:
pipeline = Pipeline([
    ('preprocessor', TextPreprocessor()),
    ('vectorizer', Word2VecVectorizer())
])

In [8]:
pipeline.fit(X_train)

Pipeline(steps=[('preprocessor', TextPreprocessor()),
                ('vectorizer', Word2VecVectorizer())])

In [9]:
X_train_embeddings = pipeline.transform(X_train)

In [10]:
X_test_embeddings = pipeline.transform(X_test)

## model selection

### RF

In [13]:
rf = RandomForestClassifier()
rf.fit(X_train_embeddings, y_train)

RandomForestClassifier()

In [16]:
# leaves unlimited - train accuracy is 1.0
rf_pred_train = rf.predict(X_train_embeddings)
print(f'Accuracy: {accuracy_score(y_train, rf_pred_train)}')
print(classification_report(y_train, rf_pred_train))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     21852
           1       1.00      1.00      1.00     30361

    accuracy                           1.00     52213
   macro avg       1.00      1.00      1.00     52213
weighted avg       1.00      1.00      1.00     52213



In [14]:
# random forest f1 0.98
rf_pred = rf.predict(X_test_embeddings)
print(f'Accuracy: {accuracy_score(y_test, rf_pred)}')
print(classification_report(y_test, rf_pred))

Accuracy: 0.980312547878045
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      5518
           1       0.99      0.97      0.98      7536

    accuracy                           0.98     13054
   macro avg       0.98      0.98      0.98     13054
weighted avg       0.98      0.98      0.98     13054



### Grid search RF

In [27]:
rf_2 = RandomForestClassifier()

params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_1 = GridSearchCV(rf_2, params, cv=3, scoring='f1', verbose=3)

In [28]:
grid_1.fit(X_train_embeddings, y_train)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
[CV 1/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=50;, score=0.951 total time=  10.5s
[CV 2/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=50;, score=0.952 total time=  10.4s
[CV 3/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=50;, score=0.953 total time=  11.2s
[CV 1/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.951 total time=  20.9s
[CV 2/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.948 total time=  20.8s
[CV 3/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.952 total time=  20.8s
[CV 1/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=150;, score=0.953 total time=  31.1s
[CV 2/3] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=150;, score=0.952 total time=  32.7s
[CV 3

[CV 2/3] END max_depth=5, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.950 total time=  31.0s
[CV 3/3] END max_depth=5, min_samples_leaf=4, min_samples_split=5, n_estimators=150;, score=0.951 total time=  31.0s
[CV 1/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=50;, score=0.948 total time=  10.3s
[CV 2/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=50;, score=0.950 total time=  10.3s
[CV 3/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=50;, score=0.948 total time=  10.3s
[CV 1/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=100;, score=0.950 total time=  20.6s
[CV 2/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=100;, score=0.951 total time=  20.7s
[CV 3/3] END max_depth=5, min_samples_leaf=4, min_samples_split=10, n_estimators=100;, score=0.953 total time=  20.7s
[CV 1/3] END max_depth=5, min_samples_leaf=4, min_samples_spl

[CV 3/3] END max_depth=10, min_samples_leaf=4, min_samples_split=2, n_estimators=100;, score=0.972 total time=  39.3s
[CV 1/3] END max_depth=10, min_samples_leaf=4, min_samples_split=2, n_estimators=150;, score=0.973 total time=  55.5s
[CV 2/3] END max_depth=10, min_samples_leaf=4, min_samples_split=2, n_estimators=150;, score=0.972 total time=  55.2s
[CV 3/3] END max_depth=10, min_samples_leaf=4, min_samples_split=2, n_estimators=150;, score=0.972 total time=  55.1s
[CV 1/3] END max_depth=10, min_samples_leaf=4, min_samples_split=5, n_estimators=50;, score=0.972 total time=  18.5s
[CV 2/3] END max_depth=10, min_samples_leaf=4, min_samples_split=5, n_estimators=50;, score=0.971 total time=  18.4s
[CV 3/3] END max_depth=10, min_samples_leaf=4, min_samples_split=5, n_estimators=50;, score=0.973 total time=  18.5s
[CV 1/3] END max_depth=10, min_samples_leaf=4, min_samples_split=5, n_estimators=100;, score=0.973 total time=  36.9s
[CV 2/3] END max_depth=10, min_samples_leaf=4, min_samples_

[CV 1/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=100;, score=0.980 total time=  49.8s
[CV 2/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=100;, score=0.977 total time=  50.2s
[CV 3/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=100;, score=0.978 total time=  49.3s
[CV 1/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=150;, score=0.979 total time= 1.3min
[CV 2/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=150;, score=0.978 total time= 1.3min
[CV 3/3] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=150;, score=0.978 total time= 1.3min
[CV 1/3] END max_depth=20, min_samples_leaf=4, min_samples_split=2, n_estimators=50;, score=0.978 total time=  24.8s
[CV 2/3] END max_depth=20, min_samples_leaf=4, min_samples_split=2, n_estimators=50;, score=0.976 total time=  25.4s
[CV 3/3] END max_depth=20, min_samples_leaf=4, min_s

GridSearchCV(cv=3, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 150]},
             scoring='f1', verbose=3)

In [31]:
grid_1.best_estimator_, grid_1.best_params_

(RandomForestClassifier(max_depth=20, n_estimators=150),
 {'max_depth': 20,
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 150})

In [32]:
best_rf = grid_1.best_estimator_
best_rf.fit(X_train_embeddings, y_train)

RandomForestClassifier(max_depth=20, n_estimators=150)

In [34]:
best_rf_pred = best_rf.predict(X_test_embeddings)
print(f'Accuracy: {accuracy_score(y_test, best_rf_pred)}')
print(classification_report(y_test, best_rf_pred))

Accuracy: 0.9784740309483683
              precision    recall  f1-score   support

           0       0.96      0.99      0.97      5518
           1       0.99      0.97      0.98      7536

    accuracy                           0.98     13054
   macro avg       0.98      0.98      0.98     13054
weighted avg       0.98      0.98      0.98     13054



### lightgbm

In [39]:
lgbm = LGBMClassifier()
lgbm.fit(X_train_embeddings, y_train)

[LightGBM] [Info] Number of positive: 30361, number of negative: 21852
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013832 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25500
[LightGBM] [Info] Number of data points in the train set: 52213, number of used features: 100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.581484 -> initscore=0.328866
[LightGBM] [Info] Start training from score 0.328866


LGBMClassifier()

In [40]:
lgbm_pred = lgbm.predict(X_test_embeddings)
print(f'Accuracy: {accuracy_score(y_test, lgbm_pred)}')
print(classification_report(y_test, lgbm_pred))

Accuracy: 0.983683162249119
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      5518
           1       0.99      0.98      0.99      7536

    accuracy                           0.98     13054
   macro avg       0.98      0.98      0.98     13054
weighted avg       0.98      0.98      0.98     13054



### Grid search lightgbm

In [46]:
lgbm_2 = LGBMClassifier(verbose=0)

params_2 = {
    'num_leaves': [15, 31, 45],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'max_depth': [-1, 10, 20],
    'min_data_in_leaf': [10, 20, 30]
}

grid_2 = GridSearchCV(lgbm_2, params_2, cv=3, verbose=3)

In [47]:
grid_2.fit(X_train_embeddings, y_train)

Fitting 3 folds for each of 243 candidates, totalling 729 fits
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 1/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=10, n_estimators=100, num_leaves=15;, score=0.936 total time=   1.2s
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 2/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=10, n_es

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 2/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=10, n_estimators=200, num_leaves=45;, score=0.963 total time=   3.1s
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 3/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=10, n_estimators=200, num_leaves=45;, score=0.965 total time=   2.9s
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[L

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 1/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=20, n_estimators=100, num_leaves=45;, score=0.953 total time=   1.7s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 2/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=20, n_estimators=100, num_leaves=45;, score=0.954 total time=   1.4s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[L

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 3/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=20, n_estimators=300, num_leaves=31;, score=0.968 total time=   3.5s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 1/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=20, n_estimators=300, num_leaves=45;, score=0.971 total time=   4.0s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[L

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[CV 2/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=30, n_estimators=200, num_leaves=31;, score=0.961 total time=   2.7s
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[CV 3/3] END learning_rate=0.01, max_depth=-1, min_data_in_leaf=30, n_estimators=200, num_leaves=31;, score=0.961 total time=   2.7s
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[L

...

[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 3/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=10, n_estimators=200, num_leaves=31;, score=0.985 total time=   2.1s
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[CV 1/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=10, n_estimators=200, num_leaves=45;, score=0.985 total time=   2.7s
[LightGBM] [Warning] min_data_in_leaf is set=10, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=10
[Lig

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 2/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=20, n_estimators=100, num_leaves=31;, score=0.981 total time=   1.3s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 3/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=20, n_estimators=100, num_leaves=31;, score=0.982 total time=   1.2s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[Lig

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 1/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=20, n_estimators=300, num_leaves=31;, score=0.986 total time=   2.9s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[CV 2/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=20, n_estimators=300, num_leaves=31;, score=0.985 total time=   2.8s
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[Lig

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[CV 3/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=30, n_estimators=200, num_leaves=15;, score=0.983 total time=   1.5s
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[CV 1/3] END learning_rate=0.1, max_depth=20, min_data_in_leaf=30, n_estimators=200, num_leaves=31;, score=0.985 total time=   2.4s
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[Lig

GridSearchCV(cv=3, estimator=LGBMClassifier(verbose=0),
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [-1, 10, 20],
                         'min_data_in_leaf': [10, 20, 30],
                         'n_estimators': [100, 200, 300],
                         'num_leaves': [15, 31, 45]},
             verbose=3)

In [48]:
grid_2.best_estimator_, grid_2.best_params_

(LGBMClassifier(max_depth=10, min_data_in_leaf=30, n_estimators=300, verbose=0),
 {'learning_rate': 0.1,
  'max_depth': 10,
  'min_data_in_leaf': 30,
  'n_estimators': 300,
  'num_leaves': 31})

In [49]:
best_lightgbm = grid_2.best_estimator_
best_lightgbm.fit(X_train_embeddings, y_train)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30


LGBMClassifier(max_depth=10, min_data_in_leaf=30, n_estimators=300, verbose=0)

In [50]:
best_lgbm_pred = best_lightgbm.predict(X_test_embeddings)
print(f'Accuracy: {accuracy_score(y_test, best_lgbm_pred)}')
print(classification_report(y_test, best_lgbm_pred))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.9878964302129616
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      5518
           1       0.99      0.99      0.99      7536

    accuracy                           0.99     13054
   macro avg       0.99      0.99      0.99     13054
weighted avg       0.99      0.99      0.99     13054



### CatBoost

In [51]:
cb_1 = CatBoostClassifier(
    logging_level='silent'
)

In [53]:
cb_1.fit(
    X_train_embeddings, y_train,
    eval_set=(X_test_embeddings, y_test),
    logging_level='Verbose'
)

Learning rate set to 0.084178
0:	learn: 0.5952936	test: 0.5949512	best: 0.5949512 (0)	total: 94.8ms	remaining: 1m 34s
1:	learn: 0.5073790	test: 0.5073608	best: 0.5073608 (1)	total: 117ms	remaining: 58.6s
2:	learn: 0.4454901	test: 0.4455081	best: 0.4455081 (2)	total: 132ms	remaining: 43.8s
3:	learn: 0.3936502	test: 0.3939501	best: 0.3939501 (3)	total: 145ms	remaining: 36s
4:	learn: 0.3525846	test: 0.3527970	best: 0.3527970 (4)	total: 159ms	remaining: 31.7s
5:	learn: 0.3244598	test: 0.3242113	best: 0.3242113 (5)	total: 173ms	remaining: 28.7s
6:	learn: 0.3039991	test: 0.3034475	best: 0.3034475 (6)	total: 187ms	remaining: 26.5s
7:	learn: 0.2791796	test: 0.2784589	best: 0.2784589 (7)	total: 205ms	remaining: 25.4s
8:	learn: 0.2628114	test: 0.2620312	best: 0.2620312 (8)	total: 223ms	remaining: 24.5s
9:	learn: 0.2481886	test: 0.2474484	best: 0.2474484 (9)	total: 239ms	remaining: 23.6s
10:	learn: 0.2375514	test: 0.2365131	best: 0.2365131 (10)	total: 253ms	remaining: 22.8s
11:	learn: 0.2272443	t

97:	learn: 0.0682580	test: 0.0717396	best: 0.0717396 (97)	total: 2.5s	remaining: 23s
98:	learn: 0.0679076	test: 0.0714439	best: 0.0714439 (98)	total: 2.52s	remaining: 23s
99:	learn: 0.0675320	test: 0.0711549	best: 0.0711549 (99)	total: 2.55s	remaining: 22.9s
100:	learn: 0.0671305	test: 0.0708293	best: 0.0708293 (100)	total: 2.57s	remaining: 22.9s
101:	learn: 0.0667744	test: 0.0705560	best: 0.0705560 (101)	total: 2.59s	remaining: 22.8s
102:	learn: 0.0664034	test: 0.0703200	best: 0.0703200 (102)	total: 2.61s	remaining: 22.7s
103:	learn: 0.0660792	test: 0.0700572	best: 0.0700572 (103)	total: 2.63s	remaining: 22.7s
104:	learn: 0.0656919	test: 0.0697372	best: 0.0697372 (104)	total: 2.66s	remaining: 22.7s
105:	learn: 0.0653611	test: 0.0695298	best: 0.0695298 (105)	total: 2.68s	remaining: 22.6s
106:	learn: 0.0649275	test: 0.0692063	best: 0.0692063 (106)	total: 2.7s	remaining: 22.6s
107:	learn: 0.0645508	test: 0.0688198	best: 0.0688198 (107)	total: 2.73s	remaining: 22.5s
108:	learn: 0.0641887	

193:	learn: 0.0441713	test: 0.0539762	best: 0.0539762 (193)	total: 5.21s	remaining: 21.6s
194:	learn: 0.0439792	test: 0.0538343	best: 0.0538343 (194)	total: 5.23s	remaining: 21.6s
195:	learn: 0.0438593	test: 0.0537822	best: 0.0537822 (195)	total: 5.25s	remaining: 21.5s
196:	learn: 0.0437002	test: 0.0536254	best: 0.0536254 (196)	total: 5.27s	remaining: 21.5s
197:	learn: 0.0436213	test: 0.0536072	best: 0.0536072 (197)	total: 5.29s	remaining: 21.4s
198:	learn: 0.0434254	test: 0.0534281	best: 0.0534281 (198)	total: 5.31s	remaining: 21.4s
199:	learn: 0.0433127	test: 0.0533784	best: 0.0533784 (199)	total: 5.33s	remaining: 21.3s
200:	learn: 0.0430688	test: 0.0531404	best: 0.0531404 (200)	total: 5.35s	remaining: 21.3s
201:	learn: 0.0428439	test: 0.0529802	best: 0.0529802 (201)	total: 5.37s	remaining: 21.2s
202:	learn: 0.0426935	test: 0.0528912	best: 0.0528912 (202)	total: 5.39s	remaining: 21.2s
203:	learn: 0.0425789	test: 0.0528524	best: 0.0528524 (203)	total: 5.42s	remaining: 21.1s
204:	learn

291:	learn: 0.0325981	test: 0.0466094	best: 0.0466094 (291)	total: 7.33s	remaining: 17.8s
292:	learn: 0.0325207	test: 0.0465453	best: 0.0465453 (292)	total: 7.36s	remaining: 17.8s
293:	learn: 0.0324309	test: 0.0465169	best: 0.0465169 (293)	total: 7.38s	remaining: 17.7s
294:	learn: 0.0323641	test: 0.0464958	best: 0.0464958 (294)	total: 7.39s	remaining: 17.7s
295:	learn: 0.0323121	test: 0.0464713	best: 0.0464713 (295)	total: 7.41s	remaining: 17.6s
296:	learn: 0.0322024	test: 0.0464532	best: 0.0464532 (296)	total: 7.43s	remaining: 17.6s
297:	learn: 0.0320813	test: 0.0463816	best: 0.0463816 (297)	total: 7.45s	remaining: 17.5s
298:	learn: 0.0319710	test: 0.0462860	best: 0.0462860 (298)	total: 7.47s	remaining: 17.5s
299:	learn: 0.0319124	test: 0.0462403	best: 0.0462403 (299)	total: 7.48s	remaining: 17.5s
300:	learn: 0.0317794	test: 0.0461787	best: 0.0461787 (300)	total: 7.5s	remaining: 17.4s
301:	learn: 0.0317180	test: 0.0461274	best: 0.0461274 (301)	total: 7.52s	remaining: 17.4s
302:	learn:

393:	learn: 0.0248858	test: 0.0424736	best: 0.0424736 (393)	total: 9.46s	remaining: 14.6s
394:	learn: 0.0248186	test: 0.0424477	best: 0.0424477 (394)	total: 9.48s	remaining: 14.5s
395:	learn: 0.0247913	test: 0.0424209	best: 0.0424209 (395)	total: 9.51s	remaining: 14.5s
396:	learn: 0.0247390	test: 0.0423883	best: 0.0423883 (396)	total: 9.53s	remaining: 14.5s
397:	learn: 0.0246885	test: 0.0423556	best: 0.0423556 (397)	total: 9.55s	remaining: 14.4s
398:	learn: 0.0246334	test: 0.0423397	best: 0.0423397 (398)	total: 9.57s	remaining: 14.4s
399:	learn: 0.0245973	test: 0.0423334	best: 0.0423334 (399)	total: 9.59s	remaining: 14.4s
400:	learn: 0.0245666	test: 0.0423368	best: 0.0423334 (399)	total: 9.62s	remaining: 14.4s
401:	learn: 0.0244914	test: 0.0422991	best: 0.0422991 (401)	total: 9.64s	remaining: 14.3s
402:	learn: 0.0244466	test: 0.0422561	best: 0.0422561 (402)	total: 9.66s	remaining: 14.3s
403:	learn: 0.0244025	test: 0.0422128	best: 0.0422128 (403)	total: 9.68s	remaining: 14.3s
404:	learn

...

580:	learn: 0.0168762	test: 0.0380841	best: 0.0380841 (580)	total: 13.5s	remaining: 9.7s
581:	learn: 0.0168552	test: 0.0380822	best: 0.0380822 (581)	total: 13.5s	remaining: 9.68s
582:	learn: 0.0168421	test: 0.0380757	best: 0.0380757 (582)	total: 13.5s	remaining: 9.66s
583:	learn: 0.0168113	test: 0.0380417	best: 0.0380417 (583)	total: 13.5s	remaining: 9.63s
584:	learn: 0.0167541	test: 0.0380270	best: 0.0380270 (584)	total: 13.5s	remaining: 9.61s
585:	learn: 0.0166950	test: 0.0379875	best: 0.0379875 (585)	total: 13.6s	remaining: 9.59s
586:	learn: 0.0166427	test: 0.0379299	best: 0.0379299 (586)	total: 13.6s	remaining: 9.56s
587:	learn: 0.0166202	test: 0.0379292	best: 0.0379292 (587)	total: 13.6s	remaining: 9.54s
588:	learn: 0.0165828	test: 0.0379257	best: 0.0379257 (588)	total: 13.6s	remaining: 9.52s
589:	learn: 0.0165626	test: 0.0379283	best: 0.0379257 (588)	total: 13.7s	remaining: 9.49s
590:	learn: 0.0165236	test: 0.0378945	best: 0.0378945 (590)	total: 13.7s	remaining: 9.46s
591:	learn:

675:	learn: 0.0141556	test: 0.0367176	best: 0.0367176 (675)	total: 15.6s	remaining: 7.46s
676:	learn: 0.0141421	test: 0.0367150	best: 0.0367150 (676)	total: 15.6s	remaining: 7.44s
677:	learn: 0.0141421	test: 0.0367150	best: 0.0367150 (677)	total: 15.6s	remaining: 7.41s
678:	learn: 0.0141420	test: 0.0367150	best: 0.0367150 (678)	total: 15.6s	remaining: 7.38s
679:	learn: 0.0141419	test: 0.0367150	best: 0.0367150 (679)	total: 15.6s	remaining: 7.36s
680:	learn: 0.0141418	test: 0.0367149	best: 0.0367149 (680)	total: 15.7s	remaining: 7.33s
681:	learn: 0.0141418	test: 0.0367149	best: 0.0367149 (681)	total: 15.7s	remaining: 7.31s
682:	learn: 0.0141418	test: 0.0367148	best: 0.0367148 (682)	total: 15.7s	remaining: 7.28s
683:	learn: 0.0141229	test: 0.0366977	best: 0.0366977 (683)	total: 15.7s	remaining: 7.25s
684:	learn: 0.0140924	test: 0.0366677	best: 0.0366677 (684)	total: 15.7s	remaining: 7.23s
685:	learn: 0.0140839	test: 0.0366592	best: 0.0366592 (685)	total: 15.7s	remaining: 7.2s
686:	learn:

775:	learn: 0.0134277	test: 0.0362392	best: 0.0362392 (775)	total: 17.2s	remaining: 4.97s
776:	learn: 0.0134277	test: 0.0362392	best: 0.0362392 (776)	total: 17.2s	remaining: 4.94s
777:	learn: 0.0134277	test: 0.0362392	best: 0.0362392 (776)	total: 17.2s	remaining: 4.92s
778:	learn: 0.0134276	test: 0.0362391	best: 0.0362391 (778)	total: 17.3s	remaining: 4.89s
779:	learn: 0.0134276	test: 0.0362391	best: 0.0362391 (778)	total: 17.3s	remaining: 4.87s
780:	learn: 0.0134146	test: 0.0362233	best: 0.0362233 (780)	total: 17.3s	remaining: 4.85s
781:	learn: 0.0134146	test: 0.0362233	best: 0.0362233 (781)	total: 17.3s	remaining: 4.82s
782:	learn: 0.0134146	test: 0.0362232	best: 0.0362232 (782)	total: 17.3s	remaining: 4.8s
783:	learn: 0.0134145	test: 0.0362232	best: 0.0362232 (783)	total: 17.3s	remaining: 4.77s
784:	learn: 0.0134145	test: 0.0362232	best: 0.0362232 (784)	total: 17.3s	remaining: 4.75s
785:	learn: 0.0134145	test: 0.0362232	best: 0.0362232 (785)	total: 17.4s	remaining: 4.73s
786:	learn:

879:	learn: 0.0126950	test: 0.0359606	best: 0.0359606 (879)	total: 18.9s	remaining: 2.57s
880:	learn: 0.0126950	test: 0.0359606	best: 0.0359606 (879)	total: 18.9s	remaining: 2.55s
881:	learn: 0.0126928	test: 0.0359596	best: 0.0359596 (881)	total: 18.9s	remaining: 2.53s
882:	learn: 0.0126928	test: 0.0359596	best: 0.0359596 (882)	total: 18.9s	remaining: 2.5s
883:	learn: 0.0126800	test: 0.0359732	best: 0.0359596 (882)	total: 18.9s	remaining: 2.48s
884:	learn: 0.0126800	test: 0.0359732	best: 0.0359596 (882)	total: 18.9s	remaining: 2.46s
885:	learn: 0.0126800	test: 0.0359732	best: 0.0359596 (882)	total: 18.9s	remaining: 2.44s
886:	learn: 0.0126799	test: 0.0359732	best: 0.0359596 (882)	total: 19s	remaining: 2.42s
887:	learn: 0.0126799	test: 0.0359731	best: 0.0359596 (882)	total: 19s	remaining: 2.39s
888:	learn: 0.0126799	test: 0.0359731	best: 0.0359596 (882)	total: 19s	remaining: 2.37s
889:	learn: 0.0126798	test: 0.0359731	best: 0.0359596 (882)	total: 19s	remaining: 2.35s
890:	learn: 0.01267

982:	learn: 0.0118106	test: 0.0356147	best: 0.0356147 (982)	total: 20.5s	remaining: 355ms
983:	learn: 0.0117923	test: 0.0355955	best: 0.0355955 (983)	total: 20.5s	remaining: 334ms
984:	learn: 0.0117707	test: 0.0355925	best: 0.0355925 (984)	total: 20.6s	remaining: 313ms
985:	learn: 0.0117534	test: 0.0355807	best: 0.0355807 (985)	total: 20.6s	remaining: 293ms
986:	learn: 0.0117306	test: 0.0355903	best: 0.0355807 (985)	total: 20.7s	remaining: 272ms
987:	learn: 0.0117005	test: 0.0355940	best: 0.0355807 (985)	total: 20.7s	remaining: 251ms
988:	learn: 0.0116795	test: 0.0355810	best: 0.0355807 (985)	total: 20.7s	remaining: 230ms
989:	learn: 0.0116558	test: 0.0355917	best: 0.0355807 (985)	total: 20.7s	remaining: 210ms
990:	learn: 0.0116046	test: 0.0355316	best: 0.0355316 (990)	total: 20.8s	remaining: 189ms
991:	learn: 0.0115897	test: 0.0355245	best: 0.0355245 (991)	total: 20.8s	remaining: 168ms
992:	learn: 0.0115762	test: 0.0355340	best: 0.0355245 (991)	total: 20.8s	remaining: 147ms
993:	learn

In [54]:
cb_pred = cb_1.predict(X_test_embeddings)
print(f'Accuracy: {accuracy_score(y_test, cb_pred)}')
print(classification_report(y_test, cb_pred))

Accuracy: 0.987053776620193
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      5518
           1       0.99      0.98      0.99      7536

    accuracy                           0.99     13054
   macro avg       0.99      0.99      0.99     13054
weighted avg       0.99      0.99      0.99     13054



In [59]:
best_model = best_lightgbm

#### возьмем lgbm как лучшую модель - качество сопоставимо с catboost, время обучения меньше

### load 4 datasets and try best models

In [55]:
pickl_h_train = pd.read_pickle('../data/outfox/human/train_humans.pkl')
pickl_h_test = pd.read_pickle('../data/outfox/human/test_humans.pkl')
pickl_h_valid = pd.read_pickle('../data/outfox/human/valid_humans.pkl')
df_human_test = pd.DataFrame(pickl_h_test)
df_human_train = pd.DataFrame(pickl_h_train)
df_human_valid = pd.DataFrame(pickl_h_valid)
df_human = pd.concat([df_human_train, df_human_test, df_human_valid], ignore_index=True)
df_human.rename(columns={0: 'text'}, inplace=True)
df_human['generated'] = 0
df_human

pickl_gpt_train = pd.read_pickle('../data/outfox/gpt/train_lms.pkl')
pickl_gpt_test = pd.read_pickle('../data/outfox/gpt/test_lms.pkl')
pickl_gpt_valid = pd.read_pickle('../data/outfox/gpt/valid_lms.pkl')
df_gpt_train = pd.DataFrame(pickl_gpt_train)
df_gpt_test = pd.DataFrame(pickl_gpt_test)
df_gpt_valid = pd.DataFrame(pickl_gpt_valid)
df_gpt = pd.concat([df_gpt_train, df_gpt_test, df_gpt_valid], ignore_index=True)
df_gpt.rename(columns={0: 'text'}, inplace=True)
df_gpt['generated'] = 1
df_gpt

df_1 = pd.concat([df_human, df_gpt], ignore_index=True)
df_1 = df_1.sample(30800).reset_index(drop=True)
df_1 = df_1.drop_duplicates()

filename = 'outfox_gpt_3_5_dataset.pkl'
with open(filename, 'wb') as data_file:
    pickle.dump(df_1, data_file)

In [56]:
pickl_h_train = pd.read_pickle('../data/outfox/human/train_humans.pkl')
pickl_h_test = pd.read_pickle('../data/outfox/human/test_humans.pkl')
pickl_h_valid = pd.read_pickle('../data/outfox/human/valid_humans.pkl')
df_human_test = pd.DataFrame(pickl_h_test)
df_human_train = pd.DataFrame(pickl_h_train)
df_human_valid = pd.DataFrame(pickl_h_valid)
df_human = pd.concat([df_human_train, df_human_test, df_human_valid], ignore_index=True)
df_human.rename(columns={0: 'text'}, inplace=True)
df_human['generated'] = 0
df_human

pickl_flan_train = pd.read_pickle('../data/outfox/flan_t5/train_lms_flan.pkl')
pickl_flan_test = pd.read_pickle('../data/outfox/flan_t5/test_lms_flan.pkl')
pickl_flan_valid = pd.read_pickle('../data/outfox/flan_t5/valid_lms_flan.pkl')
df_flan_train = pd.DataFrame(pickl_flan_train)
df_flan_test = pd.DataFrame(pickl_flan_test)
df_flan_valid = pd.DataFrame(pickl_flan_valid)
df_flan = pd.concat([df_flan_train, df_flan_test, df_flan_valid], ignore_index=True)
df_flan.rename(columns={0: 'text'}, inplace=True)
df_flan['generated'] = 1
df_flan

df_2 = pd.concat([df_human, df_flan], ignore_index=True)
df_2 = df_2.sample(30800).reset_index(drop=True)
df_2.drop_duplicates()

filename = 'outfox_flan_t5_dataset.pkl'
with open(filename, 'wb') as data_file:
    pickle.dump(df_2, data_file)

In [57]:
pickl_h_train = pd.read_pickle('../data/outfox/human/train_humans.pkl')
pickl_h_test = pd.read_pickle('../data/outfox/human/test_humans.pkl')
pickl_h_valid = pd.read_pickle('../data/outfox/human/valid_humans.pkl')
df_human_test = pd.DataFrame(pickl_h_test)
df_human_train = pd.DataFrame(pickl_h_train)
df_human_valid = pd.DataFrame(pickl_h_valid)
df_human = pd.concat([df_human_train, df_human_test, df_human_valid], ignore_index=True)
df_human.rename(columns={0: 'text'}, inplace=True)
df_human['generated'] = 0
df_human

pickl_davinci_train = pd.read_pickle('../data/outfox/davinci/train_lms_davinci.pkl')
pickl_davinci_test = pd.read_pickle('../data/outfox/davinci/test_lms_davinci.pkl')
pickl_davinci_valid = pd.read_pickle('../data/outfox/davinci/valid_lms_davinci.pkl')
df_davinci_train = pd.DataFrame(pickl_davinci_train)
df_davinci_test = pd.DataFrame(pickl_davinci_test)
df_davinci_valid = pd.DataFrame(pickl_davinci_valid)
df_davinci = pd.concat([df_davinci_train, df_davinci_test, df_davinci_valid], ignore_index=True)
df_davinci.rename(columns={0: 'text'}, inplace=True)
df_davinci['generated'] = 1
df_davinci

df_3 = pd.concat([df_human, df_davinci], ignore_index=True)
df_3 = df_3.sample(30800).reset_index(drop=True)
df_3.drop_duplicates()

filename = 'outfox_davinci_dataset.pkl'
with open(filename, 'wb') as data_file:
    pickle.dump(df_3, data_file)

In [75]:
df_4 = pd.read_json('../data/data_final.json')
df_4['label'] = df_4['label'].map({'human_text': 0, 'mixed_text': 1, 'machine_text': 2})

In [82]:
X_4 = df_4['text']
y_4 = df_4['label']

pipeline_2 = Pipeline([
    ('preprocessor', TextPreprocessor()),
    ('vectorizer', Word2VecVectorizer())
])

In [83]:
Xtrain4, Xtest4, ytrain4, ytest4 = train_test_split(X_4, y_4, test_size=0.2, random_state=42)

In [84]:
pipeline_2.fit(Xtrain4)

Pipeline(steps=[('preprocessor', TextPreprocessor()),
                ('vectorizer', Word2VecVectorizer())])

In [85]:
X_train4_embeddings = pipeline_2.transform(Xtrain4)

In [86]:
X_test4_embeddings = pipeline_2.transform(Xtest4)

In [88]:
best_rf_2 = grid_1.best_estimator_
best_rf_2.fit(X_train4_embeddings, ytrain4)

RandomForestClassifier(max_depth=20, n_estimators=150)

In [89]:
pred_best_rf_df_4 = best_rf_2.predict(X_test4_embeddings)
print(f'Accuracy: {accuracy_score(ytest4, pred_best_rf_df_4)}')
print(classification_report(ytest4, pred_best_rf_df_4))

Accuracy: 0.7762303485987696
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      3835
           1       0.67      0.66      0.67      3917
           2       0.71      0.67      0.69      3952

    accuracy                           0.78     11704
   macro avg       0.77      0.78      0.77     11704
weighted avg       0.77      0.78      0.77     11704



In [90]:
best_lgbm_2 = grid_2.best_estimator_
best_lgbm_2.fit(X_train4_embeddings, ytrain4)

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30


LGBMClassifier(max_depth=10, min_data_in_leaf=30, n_estimators=300, verbose=0)

In [91]:
pred_best_lgbm_df_4 = best_lgbm_2.predict(X_test4_embeddings)
print(f'Accuracy: {accuracy_score(ytest4, pred_best_lgbm_df_4)}')
print(classification_report(ytest4, pred_best_lgbm_df_4))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.8155331510594669
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      3835
           1       0.74      0.71      0.72      3917
           2       0.76      0.74      0.75      3952

    accuracy                           0.82     11704
   macro avg       0.81      0.82      0.81     11704
weighted avg       0.81      0.82      0.81     11704



In [92]:
cb_2 = CatBoostClassifier(
    logging_level='silent'
)

In [93]:
cb_2.fit(
    X_train4_embeddings, ytrain4,
    eval_set=(X_test4_embeddings, ytest4),
    logging_level='Verbose'
)

Learning rate set to 0.11661
0:	learn: 1.0514042	test: 1.0524364	best: 1.0524364 (0)	total: 75.1ms	remaining: 1m 15s
1:	learn: 1.0131465	test: 1.0146977	best: 1.0146977 (1)	total: 100ms	remaining: 50s
2:	learn: 0.9801711	test: 0.9821159	best: 0.9821159 (2)	total: 125ms	remaining: 41.6s
3:	learn: 0.9508952	test: 0.9535675	best: 0.9535675 (3)	total: 154ms	remaining: 38.3s
4:	learn: 0.9258418	test: 0.9290471	best: 0.9290471 (4)	total: 181ms	remaining: 36.1s
5:	learn: 0.9053170	test: 0.9089790	best: 0.9089790 (5)	total: 217ms	remaining: 35.9s
6:	learn: 0.8880047	test: 0.8915860	best: 0.8915860 (6)	total: 246ms	remaining: 34.9s
7:	learn: 0.8710746	test: 0.8753430	best: 0.8753430 (7)	total: 275ms	remaining: 34.1s
8:	learn: 0.8572814	test: 0.8620133	best: 0.8620133 (8)	total: 307ms	remaining: 33.8s
9:	learn: 0.8440480	test: 0.8488408	best: 0.8488408 (9)	total: 337ms	remaining: 33.4s
10:	learn: 0.8318862	test: 0.8371972	best: 0.8371972 (10)	total: 380ms	remaining: 34.2s
11:	learn: 0.8206138	te

97:	learn: 0.5839160	test: 0.6076516	best: 0.6076516 (97)	total: 4.65s	remaining: 42.8s
98:	learn: 0.5826909	test: 0.6066368	best: 0.6066368 (98)	total: 4.7s	remaining: 42.7s
99:	learn: 0.5814958	test: 0.6055859	best: 0.6055859 (99)	total: 4.75s	remaining: 42.7s
100:	learn: 0.5799090	test: 0.6040656	best: 0.6040656 (100)	total: 4.82s	remaining: 42.9s
101:	learn: 0.5790604	test: 0.6033230	best: 0.6033230 (101)	total: 4.88s	remaining: 43s
102:	learn: 0.5778418	test: 0.6022805	best: 0.6022805 (102)	total: 4.93s	remaining: 43s
103:	learn: 0.5763048	test: 0.6010978	best: 0.6010978 (103)	total: 5.01s	remaining: 43.2s
104:	learn: 0.5753559	test: 0.6001815	best: 0.6001815 (104)	total: 5.07s	remaining: 43.2s
105:	learn: 0.5742475	test: 0.5993896	best: 0.5993896 (105)	total: 5.13s	remaining: 43.3s
106:	learn: 0.5727032	test: 0.5980991	best: 0.5980991 (106)	total: 5.2s	remaining: 43.4s
107:	learn: 0.5713228	test: 0.5970066	best: 0.5970066 (107)	total: 5.26s	remaining: 43.5s
108:	learn: 0.5699267	

191:	learn: 0.4918450	test: 0.5351103	best: 0.5351103 (191)	total: 9.4s	remaining: 39.5s
192:	learn: 0.4912024	test: 0.5347166	best: 0.5347166 (192)	total: 9.45s	remaining: 39.5s
193:	learn: 0.4904634	test: 0.5339873	best: 0.5339873 (193)	total: 9.51s	remaining: 39.5s
194:	learn: 0.4898642	test: 0.5335751	best: 0.5335751 (194)	total: 9.57s	remaining: 39.5s
195:	learn: 0.4892005	test: 0.5331313	best: 0.5331313 (195)	total: 9.62s	remaining: 39.5s
196:	learn: 0.4886067	test: 0.5327628	best: 0.5327628 (196)	total: 9.66s	remaining: 39.4s
197:	learn: 0.4878996	test: 0.5321605	best: 0.5321605 (197)	total: 9.71s	remaining: 39.3s
198:	learn: 0.4872132	test: 0.5318017	best: 0.5318017 (198)	total: 9.76s	remaining: 39.3s
199:	learn: 0.4866448	test: 0.5313499	best: 0.5313499 (199)	total: 9.81s	remaining: 39.2s
200:	learn: 0.4859870	test: 0.5307512	best: 0.5307512 (200)	total: 9.85s	remaining: 39.2s
201:	learn: 0.4851644	test: 0.5302232	best: 0.5302232 (201)	total: 9.9s	remaining: 39.1s
202:	learn: 

287:	learn: 0.4350632	test: 0.4957318	best: 0.4957318 (287)	total: 13.9s	remaining: 34.5s
288:	learn: 0.4345468	test: 0.4954040	best: 0.4954040 (288)	total: 14s	remaining: 34.5s
289:	learn: 0.4341227	test: 0.4950588	best: 0.4950588 (289)	total: 14.1s	remaining: 34.4s
290:	learn: 0.4338482	test: 0.4949491	best: 0.4949491 (290)	total: 14.1s	remaining: 34.3s
291:	learn: 0.4333557	test: 0.4945604	best: 0.4945604 (291)	total: 14.1s	remaining: 34.3s
292:	learn: 0.4328362	test: 0.4943429	best: 0.4943429 (292)	total: 14.2s	remaining: 34.2s
293:	learn: 0.4325496	test: 0.4942068	best: 0.4942068 (293)	total: 14.2s	remaining: 34.1s
294:	learn: 0.4321819	test: 0.4940084	best: 0.4940084 (294)	total: 14.3s	remaining: 34.1s
295:	learn: 0.4317232	test: 0.4936924	best: 0.4936924 (295)	total: 14.3s	remaining: 34s
296:	learn: 0.4312506	test: 0.4934277	best: 0.4934277 (296)	total: 14.3s	remaining: 34s
297:	learn: 0.4307420	test: 0.4931309	best: 0.4931309 (297)	total: 14.4s	remaining: 33.9s
298:	learn: 0.43

382:	learn: 0.3962123	test: 0.4728204	best: 0.4728204 (382)	total: 17.8s	remaining: 28.7s
383:	learn: 0.3959580	test: 0.4727247	best: 0.4727247 (383)	total: 17.8s	remaining: 28.6s
384:	learn: 0.3954640	test: 0.4724513	best: 0.4724513 (384)	total: 17.9s	remaining: 28.6s
385:	learn: 0.3950842	test: 0.4723148	best: 0.4723148 (385)	total: 17.9s	remaining: 28.5s
386:	learn: 0.3947211	test: 0.4720367	best: 0.4720367 (386)	total: 18s	remaining: 28.4s
387:	learn: 0.3942107	test: 0.4718065	best: 0.4718065 (387)	total: 18s	remaining: 28.4s
388:	learn: 0.3938942	test: 0.4716267	best: 0.4716267 (388)	total: 18s	remaining: 28.3s
389:	learn: 0.3934194	test: 0.4713374	best: 0.4713374 (389)	total: 18.1s	remaining: 28.3s
390:	learn: 0.3929511	test: 0.4709413	best: 0.4709413 (390)	total: 18.1s	remaining: 28.2s
391:	learn: 0.3926510	test: 0.4707127	best: 0.4707127 (391)	total: 18.2s	remaining: 28.2s
392:	learn: 0.3921293	test: 0.4703015	best: 0.4703015 (392)	total: 18.2s	remaining: 28.1s
393:	learn: 0.39

...

571:	learn: 0.3431273	test: 0.4464387	best: 0.4464387 (571)	total: 26.4s	remaining: 19.8s
572:	learn: 0.3427852	test: 0.4462343	best: 0.4462343 (572)	total: 26.5s	remaining: 19.7s
573:	learn: 0.3424825	test: 0.4460863	best: 0.4460863 (573)	total: 26.5s	remaining: 19.7s
574:	learn: 0.3423338	test: 0.4460251	best: 0.4460251 (574)	total: 26.6s	remaining: 19.7s
575:	learn: 0.3420390	test: 0.4458344	best: 0.4458344 (575)	total: 26.7s	remaining: 19.6s
576:	learn: 0.3418049	test: 0.4457722	best: 0.4457722 (576)	total: 26.7s	remaining: 19.6s
577:	learn: 0.3415906	test: 0.4457781	best: 0.4457722 (576)	total: 26.8s	remaining: 19.6s
578:	learn: 0.3413894	test: 0.4456293	best: 0.4456293 (578)	total: 26.8s	remaining: 19.5s
579:	learn: 0.3411062	test: 0.4454977	best: 0.4454977 (579)	total: 26.9s	remaining: 19.5s
580:	learn: 0.3409281	test: 0.4454391	best: 0.4454391 (580)	total: 26.9s	remaining: 19.4s
581:	learn: 0.3407093	test: 0.4453034	best: 0.4453034 (581)	total: 27s	remaining: 19.4s
582:	learn: 

665:	learn: 0.3223171	test: 0.4380459	best: 0.4380459 (665)	total: 31.3s	remaining: 15.7s
666:	learn: 0.3221209	test: 0.4380087	best: 0.4380087 (666)	total: 31.4s	remaining: 15.7s
667:	learn: 0.3219812	test: 0.4380364	best: 0.4380087 (666)	total: 31.4s	remaining: 15.6s
668:	learn: 0.3218660	test: 0.4379597	best: 0.4379597 (668)	total: 31.5s	remaining: 15.6s
669:	learn: 0.3215604	test: 0.4377551	best: 0.4377551 (669)	total: 31.6s	remaining: 15.5s
670:	learn: 0.3213684	test: 0.4376780	best: 0.4376780 (670)	total: 31.6s	remaining: 15.5s
671:	learn: 0.3210853	test: 0.4375535	best: 0.4375535 (671)	total: 31.7s	remaining: 15.5s
672:	learn: 0.3207489	test: 0.4374006	best: 0.4374006 (672)	total: 31.7s	remaining: 15.4s
673:	learn: 0.3206201	test: 0.4373780	best: 0.4373780 (673)	total: 31.8s	remaining: 15.4s
674:	learn: 0.3204958	test: 0.4372889	best: 0.4372889 (674)	total: 31.8s	remaining: 15.3s
675:	learn: 0.3202469	test: 0.4372077	best: 0.4372077 (675)	total: 31.9s	remaining: 15.3s
676:	learn

760:	learn: 0.3042578	test: 0.4309659	best: 0.4309473 (759)	total: 37.1s	remaining: 11.7s
761:	learn: 0.3041058	test: 0.4309136	best: 0.4309136 (761)	total: 37.1s	remaining: 11.6s
762:	learn: 0.3038800	test: 0.4307162	best: 0.4307162 (762)	total: 37.2s	remaining: 11.6s
763:	learn: 0.3036694	test: 0.4306591	best: 0.4306591 (763)	total: 37.3s	remaining: 11.5s
764:	learn: 0.3035122	test: 0.4306269	best: 0.4306269 (764)	total: 37.3s	remaining: 11.5s
765:	learn: 0.3033547	test: 0.4306063	best: 0.4306063 (765)	total: 37.4s	remaining: 11.4s
766:	learn: 0.3031804	test: 0.4305551	best: 0.4305551 (766)	total: 37.4s	remaining: 11.4s
767:	learn: 0.3030173	test: 0.4304840	best: 0.4304840 (767)	total: 37.4s	remaining: 11.3s
768:	learn: 0.3027934	test: 0.4304462	best: 0.4304462 (768)	total: 37.5s	remaining: 11.3s
769:	learn: 0.3026010	test: 0.4303005	best: 0.4303005 (769)	total: 37.5s	remaining: 11.2s
770:	learn: 0.3024202	test: 0.4302508	best: 0.4302508 (770)	total: 37.6s	remaining: 11.2s
771:	learn

856:	learn: 0.2884442	test: 0.4250966	best: 0.4250966 (856)	total: 42s	remaining: 7.01s
857:	learn: 0.2882181	test: 0.4249732	best: 0.4249732 (857)	total: 42.1s	remaining: 6.96s
858:	learn: 0.2880177	test: 0.4249022	best: 0.4249022 (858)	total: 42.1s	remaining: 6.92s
859:	learn: 0.2877762	test: 0.4247191	best: 0.4247191 (859)	total: 42.2s	remaining: 6.87s
860:	learn: 0.2876503	test: 0.4247314	best: 0.4247191 (859)	total: 42.3s	remaining: 6.83s
861:	learn: 0.2875197	test: 0.4246366	best: 0.4246366 (861)	total: 42.3s	remaining: 6.78s
862:	learn: 0.2872672	test: 0.4245432	best: 0.4245432 (862)	total: 42.4s	remaining: 6.73s
863:	learn: 0.2871007	test: 0.4245515	best: 0.4245432 (862)	total: 42.5s	remaining: 6.68s
864:	learn: 0.2868924	test: 0.4244108	best: 0.4244108 (864)	total: 42.5s	remaining: 6.63s
865:	learn: 0.2866297	test: 0.4242479	best: 0.4242479 (865)	total: 42.6s	remaining: 6.59s
866:	learn: 0.2865026	test: 0.4241876	best: 0.4241876 (866)	total: 42.6s	remaining: 6.54s
867:	learn: 

948:	learn: 0.2749402	test: 0.4210637	best: 0.4210637 (948)	total: 46.6s	remaining: 2.5s
949:	learn: 0.2748401	test: 0.4210819	best: 0.4210637 (948)	total: 46.7s	remaining: 2.46s
950:	learn: 0.2746250	test: 0.4209441	best: 0.4209441 (950)	total: 46.8s	remaining: 2.41s
951:	learn: 0.2744077	test: 0.4209069	best: 0.4209069 (951)	total: 46.8s	remaining: 2.36s
952:	learn: 0.2742600	test: 0.4208357	best: 0.4208357 (952)	total: 46.9s	remaining: 2.31s
953:	learn: 0.2741502	test: 0.4207839	best: 0.4207839 (953)	total: 47s	remaining: 2.26s
954:	learn: 0.2740480	test: 0.4207133	best: 0.4207133 (954)	total: 47s	remaining: 2.22s
955:	learn: 0.2739139	test: 0.4207232	best: 0.4207133 (954)	total: 47.1s	remaining: 2.17s
956:	learn: 0.2737842	test: 0.4206399	best: 0.4206399 (956)	total: 47.2s	remaining: 2.12s
957:	learn: 0.2735321	test: 0.4204381	best: 0.4204381 (957)	total: 47.2s	remaining: 2.07s
958:	learn: 0.2733755	test: 0.4202787	best: 0.4202787 (958)	total: 47.3s	remaining: 2.02s
959:	learn: 0.2

In [94]:
cb_pred_2 = cb_2.predict(X_test4_embeddings)
print(f'Accuracy: {accuracy_score(ytest4, cb_pred_2)}')
print(classification_report(ytest4, cb_pred_2))

Accuracy: 0.8070745044429255
              precision    recall  f1-score   support

           0       0.92      1.00      0.96      3835
           1       0.73      0.69      0.71      3917
           2       0.76      0.74      0.75      3952

    accuracy                           0.81     11704
   macro avg       0.80      0.81      0.81     11704
weighted avg       0.80      0.81      0.80     11704



In [96]:
df_5 = pd.read_json('../data/data_test.json')
df_5['label'] = df_5['label'].map({'human_text': 0, 'mixed_text': 1, 'machine_text': 2})

In [97]:
X_5 = df_5['text']
y_5 = df_5['label']

In [99]:
X_5_embeddings = pipeline_2.transform(X_5)

In [100]:
pred_best_lgbm_df_5 = best_lgbm_2.predict(X_5_embeddings)
print(f'Accuracy: {accuracy_score(y_5, pred_best_lgbm_df_5)}')
print(classification_report(y_5, pred_best_lgbm_df_5))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.6200401883453799
              precision    recall  f1-score   support

           0       0.84      0.53      0.65     11123
           1       0.51      0.62      0.56     11123
           2       0.62      0.70      0.66     11097

    accuracy                           0.62     33343
   macro avg       0.66      0.62      0.62     33343
weighted avg       0.66      0.62      0.62     33343



### getting datasets and applying pipeline

In [101]:
cb_pred_3 = cb_2.predict(X_5_embeddings)
print(f'Accuracy: {accuracy_score(y_5, cb_pred_3)}')
print(classification_report(y_5, cb_pred_3))

Accuracy: 0.6300872746903398
              precision    recall  f1-score   support

           0       0.82      0.59      0.68     11123
           1       0.53      0.58      0.56     11123
           2       0.61      0.72      0.66     11097

    accuracy                           0.63     33343
   macro avg       0.65      0.63      0.63     33343
weighted avg       0.65      0.63      0.63     33343



#### GPT 3.5 model

In [61]:
X_gpt_3_5 = df_1['text']
y_gpt_3_5 = df_1['generated']

In [62]:
X_gpt_embeddings = pipeline.transform(X_gpt_3_5)

In [63]:
pred_gpt = best_model.predict(X_gpt_embeddings)
print(f'Accuracy: {accuracy_score(y_gpt_3_5, pred_gpt)}')
print(classification_report(y_gpt_3_5, pred_gpt))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.9897769819306528
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     15315
           1       1.00      0.98      0.99     15400

    accuracy                           0.99     30715
   macro avg       0.99      0.99      0.99     30715
weighted avg       0.99      0.99      0.99     30715



#### FLAN-T5 model

In [64]:
X_flan_t5 = df_2['text']
y_flan_t5 = df_2['generated']

In [65]:
X_flan_embeddings = pipeline.transform(X_flan_t5)

In [66]:
pred_flan = best_model.predict(X_flan_embeddings)
print(f'Accuracy: {accuracy_score(y_flan_t5, pred_flan)}')
print(classification_report(y_flan_t5, pred_flan))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.7134090909090909
              precision    recall  f1-score   support

           0       0.64      1.00      0.78     15400
           1       0.99      0.43      0.60     15400

    accuracy                           0.71     30800
   macro avg       0.82      0.71      0.69     30800
weighted avg       0.82      0.71      0.69     30800



#### davinci model

In [67]:
X_davinci = df_3['text']
y_davinci = df_3['generated']

In [68]:
X_davinci_embeddings = pipeline.transform(X_davinci)

In [69]:
pred_davinci = best_model.predict(X_davinci_embeddings)
print(f'Accuracy: {accuracy_score(y_davinci, pred_davinci)}')
print(classification_report(y_davinci, pred_davinci))

[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
Accuracy: 0.9800649350649351
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     15400
           1       1.00      0.96      0.98     15400

    accuracy                           0.98     30800
   macro avg       0.98      0.98      0.98     30800
weighted avg       0.98      0.98      0.98     30800



### dump model to pickle

In [70]:
model_filename = 'model_lgbm.pkl'
with open(model_filename, 'wb') as model_file:
    pickle.dump(best_model, model_file)

In [71]:
pipe_filename = 'w2v_preprocess_pipeline.pkl'
with open(pipe_filename, 'wb') as pipe_file:
    pickle.dump(pipeline, pipe_file)